# NBI + Weather Correlation AnalysisThis notebook demonstrates the Phase 5 analysis layer of the mosquito wastewater surveillance pipeline:1. Calculate the **Normalized Biomass Index (NBI)** for each target vector species2. Merge NBI values with weather data (daily high/low temperature, rainfall)3. Run correlation tests to see whether vector biomass tracks weather patternsDaily **high and low** temperatures are used instead of the daily mean, since mosquito development is driven by temperature extremes rather than the average — overnight lows affect larval/egg survival, and daytime highs affect metabolic and development rate.**Note:** This notebook uses synthetic (fake) sample data by default to demonstrate the analysis method. A real-data version using live weather data is included in Section 4b.

## 1. Imports- `pandas` / `numpy` — for building and manipulating the data tables- `scipy.stats` — for the correlation tests (Pearson and Spearman)- `matplotlib` — for visualizing the relationship- `fetch_weather_data` — our own script (in `scripts/`) for pulling real Boston weather data from Open-Meteo

In [ ]:
import pandas as pdimport numpy as npfrom scipy.stats import pearsonr, spearmanrimport matplotlib.pyplot as plt# Import our own weather-fetching script (see scripts/fetch_weather_data.py)import syssys.path.append("../scripts")from fetch_weather_data import fetch_boston_weather

## 2. Synthetic Read Count DataIn the real pipeline, this table would come from Phase 3 (alignment) output — unique mapped read counts for each species and crAssphage, per sampling date.Here we simulate 8 monthly samples (March–October) with made-up numbers that loosely trend upward with warmer months, just to have something realistic to test the method on.

In [ ]:
# Synthetic read count data (stand-in for real Phase 3 alignment output)read_counts = pd.DataFrame({    "sample_date": pd.date_range("2026-03-01", periods=8, freq="MS"),    "culex_reads": [80, 150, 260, 410, 590, 520, 340, 190],    "aegypti_reads": [2, 4, 9, 18, 30, 25, 14, 6],    "albopictus_reads": [30, 55, 95, 160, 220, 200, 130, 70],    "crassphage_reads": [9800, 10200, 9500, 11000, 10500, 9700, 10100, 9900],})read_counts

## 3. Calculate the Normalized Biomass Index (NBI)**NBI = Total Target Vector Reads / Total crAssphage Reads**Dividing by crAssphage reads corrects for dilution caused by rainfall and water-use fluctuations, since crAssphage is shed by humans at a roughly constant rate regardless of storm flow.

In [ ]:
def calculate_nbi(df):    """Add NBI columns for each target species."""    df = df.copy()    df["NBI_culex"] = df["culex_reads"] / df["crassphage_reads"]    df["NBI_aegypti"] = df["aegypti_reads"] / df["crassphage_reads"]    df["NBI_albopictus"] = df["albopictus_reads"] / df["crassphage_reads"]    return dfnbi_df = calculate_nbi(read_counts)nbi_df

## 4a. Synthetic Weather Data (default)In the real pipeline, this would be pulled live from Boston weather records (daily high/low temperature and rainfall, aggregated to match sample dates). Here we simulate a simple seasonal high/low curve and random rainfall so the notebook runs end-to-end without needing network access.

In [ ]:
weather = pd.DataFrame({    "sample_date": pd.date_range("2026-03-01", periods=8, freq="MS"),    "high_temp_f": [48, 58, 68, 78, 85, 88, 80, 65],    "low_temp_f":  [32, 38, 46, 56, 63, 66, 58, 44],    "rainfall_in": [3.2, 3.8, 3.5, 3.9, 3.1, 4.5, 4.0, 3.6],})weather

## 4b. Real Weather Data (optional — requires internet access)This cell pulls real historical daily high/low temperatures and rainfall for Boston from the Open-Meteo API, using our `fetch_boston_weather()` function from `scripts/fetch_weather_data.py`.**Skip this cell if running offline** — the synthetic `weather` DataFrame from Section 4a will be used instead. To switch to real data, uncomment the two lines below and run this cell after Section 4a; it will overwrite the `weather` variable with live data resampled to monthly sample dates.

In [ ]:
# Uncomment to fetch real data (requires internet access):# daily_weather = fetch_boston_weather("2026-03-01", "2026-10-31")## # Resample daily data down to one row per month, matching our monthly NBI samples.# # (When real per-sample dates are known from the actual sequencing run, merge on# # those exact dates instead of resampling to monthly.)# weather = (#     daily_weather.set_index("date")#     .resample("MS")#     .agg({"high_temp_f": "mean", "low_temp_f": "mean", "rainfall_in": "sum"})#     .reset_index()#     .rename(columns={"date": "sample_date"})# )# weather

## 5. Merge NBI and Weather DataJoin the two tables on `sample_date` so each row has both the biomass index and the matching weather conditions.

In [ ]:
merged = pd.merge(nbi_df, weather, on="sample_date")merged

## 6. Correlation Tests: NBI vs. High and Low TemperatureHigh and low temperature are tested **separately**, since they may relate to mosquito biomass differently:- **Low temperature** relates more to larval/egg survival (a cold snap can wipe out a generation)- **High temperature** relates more to metabolic/development rate (faster development up to a point, then heat becomes limiting)- **Pearson** correlation measures a *linear* relationship (assumes roughly normal, straight-line trend)- **Spearman** correlation measures a *monotonic* relationship (does one variable consistently rise/fall with the other, even if not perfectly linear) — more robust to outliersBoth return:- **r (correlation coefficient):** ranges from -1 to 1. Closer to 1 = strong positive relationship, closer to -1 = strong negative relationship, near 0 = no relationship- **p-value:** probability the observed relationship is due to random chance. Conventionally, p < 0.05 is considered statistically significant

In [ ]:
species = ["NBI_culex", "NBI_aegypti", "NBI_albopictus"]temp_vars = ["high_temp_f", "low_temp_f"]results = []for sp in species:    for temp in temp_vars:        r_p, p_p = pearsonr(merged[sp], merged[temp])        r_s, p_s = spearmanr(merged[sp], merged[temp])        results.append({            "species": sp,            "temp_variable": temp,            "pearson_r": round(r_p, 3),            "pearson_p": round(p_p, 3),            "spearman_r": round(r_s, 3),            "spearman_p": round(p_s, 3),        })results_df = pd.DataFrame(results)results_df

## 7. Visualize the RelationshipScatter plots comparing Culex NBI against both high and low temperature, side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].scatter(merged["high_temp_f"], merged["NBI_culex"], color="firebrick")axes[0].set_xlabel("Daily High Temp (°F)")axes[0].set_ylabel("Normalized Biomass Index (Culex)")axes[0].set_title("Culex NBI vs. High Temp")axes[1].scatter(merged["low_temp_f"], merged["NBI_culex"], color="steelblue")axes[1].set_xlabel("Daily Low Temp (°F)")axes[1].set_ylabel("Normalized Biomass Index (Culex)")axes[1].set_title("Culex NBI vs. Low Temp")plt.tight_layout()plt.show()

## 8. Next Steps (once real data is available)- Replace synthetic `read_counts` with real Phase 3 alignment output (per-sample read counts from BAM files)- Switch on Section 4b to pull real Boston weather data instead of the synthetic values in 4a- Merge on **exact sample dates** once real sequencing sample dates are known, instead of resampling weather to monthly- Consider lag-response regression (e.g. does a cold snap 1–2 weeks ago predict a drop in today's NBI, since population response to weather isn't instant)